In [7]:
import pandas as pd
import numpy as np
import argparse
import os
import random

try:
    # Argument parsing
    parser = argparse.ArgumentParser(description='Parallel XRD Simulation')
    parser.add_argument('--n_workers', type=int, help='Total number of workers')
    parser.add_argument('--worker_num', type=int, help='Current worker number (0-indexed)')
    args = parser.parse_args()

    n_workers = args.n_workers
    worker_num = args.worker_num
except:
    n_workers = 100
    worker_num = 0
    

train_df = pd.read_csv('/home/gridsan/tmackey/cdvae/data/mp_20/train.csv')
val_df = pd.read_csv('/home/gridsan/tmackey/cdvae/data/mp_20/val.csv')
test_df = pd.read_csv('/home/gridsan/tmackey/cdvae/data/mp_20/test.csv')

dataframes = {'train': train_df, 'val': val_df, 'test': test_df}

for name, data in dataframes.items():
    data['xrd_peak_locations'] = data['xrd_peak_locations'].apply(
        lambda x: [float(i) for i in x.strip('[]').split(',')]
    )

    #make sure we can read in the diffraction patterns
    data['xrd_peak_intensities'] = data['xrd_peak_intensities'].apply(
        lambda x: [float(i) for i in x.strip('[]').split(',')]
    )

def caglioti_fwhm(theta, U, V, W):
    """
    Calculate the FWHM using the Caglioti formula.
    theta: float, the angle in degrees
    U, V, W: Caglioti parameters
    """
    rad_theta = np.radians(theta / 2)  # Convert theta to radians
    return (U * np.tan(rad_theta)**2 + V * np.tan(rad_theta) + W)**0.5

def pseudo_voigt(x, center, amplitude, U, V, W, eta):
    """
    Pseudo-Voigt function using Caglioti FWHM.
    x: array-like, the independent variable
    center: float, the center of the peak
    amplitude: float, the height of the peak
    U, V, W: Caglioti parameters
    eta: float, the fraction of the Lorentzian component (0 <= eta <= 1)
    """
    fwhm = caglioti_fwhm(center, U, V, W)
    sigma = fwhm / (2 * np.sqrt(2 * np.log(2)))  # Convert FWHM to sigma for Gaussian
    lorentzian = amplitude * (fwhm**2 / ((x - center)**2 + fwhm**2))
    gaussian = amplitude * np.exp(-(x - center)**2 / (2 * sigma**2))
    return eta * lorentzian + (1 - eta) * gaussian

def superimposed_pseudo_voigt(x, xy_merge, U, V, W, eta):
    """
    Superimpose multiple pseudo-Voigt functions using Caglioti FWHM.
    x: array-like, the independent variable
    xy_merge: nx2 array, first column is peak locations, second column is intensities
    U, V, W: Caglioti parameters
    eta: float, the fraction of the Lorentzian component (0 <= eta <= 1)
    """
    total = np.zeros_like(x)
    for row in xy_merge:
        center, amplitude = row
        total += pseudo_voigt(x, center, amplitude, U, V, W, eta)
    total = total / max(total)
    return total

peak_shapes = [(0.05, -0.06, 0.07), (0.05, -0.01, 0.01),
                (0.0, 0.0, 0.01), (0.0, 0.0, random.uniform(0.001, 0.1))]

# Your modified pseudo_voigt and superimposed_pseudo_voigt functions go here

# Function to simulate XRD for each row
def simulate_pv_xrd_for_row(row_tuple, U, V, W):
    index, row = row_tuple  # Unpack the tuple

    x = np.linspace(5, 85, 8192)
    eta = 0.75  # Fraction of Lorentzian component (common for all peaks)

    # Combine peak locations and intensities into a single array
    xy_merge = np.column_stack((row['xrd_peak_locations'], row['xrd_peak_intensities']))

    sim_xrd = superimposed_pseudo_voigt(x, xy_merge, U, V, W, eta)

    return sim_xrd

def apply_simulation(data, U, V, W, worker_num, n_workers):
    # Split data for the current worker
    chunk_size = len(data) // n_workers
    start_idx = worker_num * chunk_size
    end_idx = None if worker_num == n_workers - 1 else start_idx + chunk_size # Last worker gets the rest
    worker_data = data.iloc[start_idx:end_idx]

    print("Worker", worker_num, "processing", len(worker_data), "rows")
    print("start index:", start_idx, "end index:", end_idx)

    # Process using a list comprehension
    results = [simulate_pv_xrd_for_row((idx, row), U, V, W) for idx, row in worker_data.iterrows()]
    return results

for name, data in dataframes.items():
    # Assuming peak_shapes is defined elsewhere in your code
    for peak_shape, (U, V, W) in enumerate(peak_shapes): 
        sim_pv_xrd_intensities = apply_simulation(data, U, V, W, worker_num, n_workers)

        # Save results as a numpy array
        output_filename = f'{name}_sim_pv_xrd_intensities_{peak_shape}_worker_{worker_num}.npy'
        np.save(os.path.join('/home/gridsan/tmackey/cdvae/data/mp_20', output_filename), sim_pv_xrd_intensities)

    print("Simulation completed for worker number:", worker_num)

usage: ipykernel_launcher.py [-h] [--n_workers N_WORKERS]
                             [--worker_num WORKER_NUM]
ipykernel_launcher.py: error: unrecognized arguments: --f=/home/gridsan/tmackey/.local/share/jupyter/runtime/kernel-v2-41116613qo1W7ozG4M6.json


Worker 0 processing 271 rows
start index: 0 end index: 271
Worker 0 processing 271 rows
start index: 0 end index: 271
Worker 0 processing 271 rows
start index: 0 end index: 271
Worker 0 processing 271 rows
start index: 0 end index: 271
Simulation completed for worker number: 0
Worker 0 processing 43 rows
start index: 0 end index: 43
Worker 0 processing 43 rows
start index: 0 end index: 43
Worker 0 processing 43 rows
start index: 0 end index: 43
Worker 0 processing 43 rows
start index: 0 end index: 43
Simulation completed for worker number: 0
Worker 0 processing 90 rows
start index: 0 end index: 90
Worker 0 processing 90 rows
start index: 0 end index: 90
Worker 0 processing 90 rows
start index: 0 end index: 90
Worker 0 processing 90 rows
start index: 0 end index: 90
Simulation completed for worker number: 0


In [20]:
np.array([])

array([], dtype=float64)

In [32]:
peak_shapes

[(0.05, -0.06, 0.07),
 (0.05, -0.01, 0.01),
 (0.0, 0.0, 0.01),
 (0.0, 0.0, 0.06894835941006727)]

In [4]:
import pandas as pd
import numpy as np
import argparse
import os
import random
from tqdm import tqdm
import torch

In [5]:
    

train_df = pd.read_csv('/home/gridsan/tmackey/cdvae/data/mp_20/train.csv')
val_df = pd.read_csv('/home/gridsan/tmackey/cdvae/data/mp_20/val.csv')
test_df = pd.read_csv('/home/gridsan/tmackey/cdvae/data/mp_20/test.csv')

dataframes = {'train': train_df, 'val': val_df, 'test': test_df}


In [6]:
peak_shapes = [(0.05, -0.06, 0.07), (0.05, -0.01, 0.01),
                (0.0, 0.0, 0.01), (0.0, 0.0, random.uniform(0.001, 0.1))]


In [7]:
n_workers = 100 

In [8]:
#read the data back in 
simulated_data = {'train': {}, 'val': {}, 'test': {}}

for name, data in dataframes.items():
    for peak_shape, (U, V, W) in enumerate(peak_shapes):
        results = []
        for worker_num in tqdm(range(n_workers)):
            output_filename = f'{name}_sim_pv_xrd_intensities_{peak_shape}_worker_{worker_num}.npy'
            sim_pv_xrd_intensities = np.load(os.path.join('/home/gridsan/tmackey/cdvae/data/mp_20', output_filename))
            results.append(sim_pv_xrd_intensities)
        results = np.vstack(results)
        simulated_data[name][peak_shape] = results



100%|██████████| 100/100 [00:17<00:00,  5.66it/s]


In [9]:
simulated_data['train'][0].shape

(27136, 8192)

In [10]:
for name, data in dataframes.items():
    for peak_shape, (U, V, W) in enumerate(peak_shapes):
        output_filename = f'{name}_sim_pv_xrd_intensities_{peak_shape}.pt'
        torch.save(os.path.join('/home/gridsan/tmackey/cdvae/data/mp_20', output_filename), simulated_data[name][peak_shape])

AttributeError: 'numpy.ndarray' object has no attribute 'flush'

In [ ]:
simulated_data = {'train': {}, 'val': {}, 'test': {}}

In [22]:
dataframes

{'train':        Unnamed: 0.2  Unnamed: 0.1  Unnamed: 0 material_id  \
 0                 0             0       37228  mp-1221227   
 1                 1             1       19480   mp-974729   
 2                 2             2       29624  mp-1185360   
 3                 3             3       38633  mp-1188861   
 4                 4             4       10889   mp-677272   
 ...             ...           ...         ...         ...   
 27131         27131         27131       37856   mp-568116   
 27132         27132         27132       11955   mp-865529   
 27133         27133         27133       26119  mp-1189241   
 27134         27134         27134       30556  mp-1104538   
 27135         27135         27135       32933   mp-756354   
 
        formation_energy_per_atom  band_gap pretty_formula  e_above_hull  \
 0                      -1.637460    0.2133    Na3MnCoNiO6      0.043001   
 1                      -0.314759    0.0000     Nd(Al2Cu)4      0.000000   
 2               

In [23]:
for name, data in dataframes.items():
    for peak_shape, (U, V, W) in enumerate(peak_shapes):
        output_filename = f'{name}_sim_pv_xrd_intensities_{peak_shape}.npy'
        results = np.load(os.path.join('/home/gridsan/tmackey/cdvae/data/mp_20', output_filename))
        simulated_data[name][peak_shape] = results

In [1]:
import pandas as pd
import numpy as np
import argparse
import os
import random
from tqdm import tqdm
import torch


train_df = pd.read_csv('/home/gridsan/tmackey/cdvae/data/mp_20/train.csv')
val_df = pd.read_csv('/home/gridsan/tmackey/cdvae/data/mp_20/val.csv')
test_df = pd.read_csv('/home/gridsan/tmackey/cdvae/data/mp_20/test.csv')

dataframes = {'train': train_df, 'val': val_df, 'test': test_df}

peak_shapes = [(0.05, -0.06, 0.07), (0.05, -0.01, 0.01),
                (0.0, 0.0, 0.01), (0.0, 0.0, random.uniform(0.001, 0.1))]

n_workers = 100 

simulated_data = {'train': {}, 'val': {}, 'test': {}}


In [3]:
for name, data in dataframes.items():
    for peak_shape, (U, V, W) in enumerate(peak_shapes):
        output_filename = f'{name}_sim_pv_xrd_intensities_{peak_shape}.npy'
        results = np.load(os.path.join('/home/gridsan/tmackey/cdvae/data/mp_20', output_filename))
        simulated_data[name][peak_shape] = results

ValueError: Cannot load file containing pickled data when allow_pickle=False

In [11]:
def stack_and_save_np(df, data, filename):
    tensor = torch.from_numpy(data).float()
    data = data.reshape(data.shape[0], 1, data.shape[1])
    data_dict = {}
    for i in range(len(df)):
        data_dict[df['material_id'][i]] = data[i]
    torch.save(data_dict, filename)
    print("Saved to {}".format(filename))

In [ ]:
for name, df in tqdm(dataframes.items()):
    for peak_shape, (U, V, W) in enumerate(peak_shapes):
        output_filename = f'{name}_sim_pv_xrd_intensities_{peak_shape}.npy'
        data = simulated_data[name][peak_shape]   
        stack_and_save_np(df, data, os.path.join('/home/gridsan/tmackey/cdvae/data/mp_20', output_filename))

  0%|          | 0/3 [00:00<?, ?it/s]

In [12]:
val_dict[ 'mp-754553'].shape

torch.Size([1, 8192])

In [28]:
xrd_peak_intensities_dict = torch.load('/home/gridsan/tmackey/cdvae/data/mp_20_dm/train' + "_xrd_peak_intensities_dict.pt")

In [31]:
xrd_peak_intensities_dict['mp-1221227'].shape

torch.Size([1, 256])